In [1]:
# 구글 드라이브 연동
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import json

# rag_path = '/content/drive/MyDrive/RAG/data/rag_chunks.json'

# # 문서 로딩
# rag_docs = []
# with open(rag_path, 'r', encoding='utf-8') as f:
#     for line in f:
#         doc = json.loads(line)
#         rag_docs.append(doc)

# print(f"총 {len(rag_docs)}개의 문서를 불러왔습니다.")
# print("예시 문서:")
# print(rag_docs[0])



총 8026개의 문서를 불러왔습니다.
예시 문서:
{'id': 'entry_0', 'context': '[질문] 지금 제일 가보고 싶은 회사는\n[답변] 제가 이 회사에 지원한 이유는 지금 가장 가고 싶은 회사가 이 회사이기 때문입니다. 이 회사에서 근무를 하면서 더 많은 능력을 키우고 더 많은 경험을 통해서 제가 가고 싶었던 이 회사 회사를 회사에 좀 더 기여할 수 있도록 많은 노력을 하려고 하고 있습니다.', 'metadata': {'occupation': 'ARD', 'gender': 'FEMALE', 'experience': 'NEW', 'date': '20230116', 'emotion': ['p-interest', 'p-interest'], 'intent_category': ['etc'], 'summary': '제가 이 회사에 지원한 이유는 가장 가고 싶은 회사이기 때문입니다. 회사에서 근무하며 회사에 기여할 수 있도록 노력을 하겠습니다.'}}


In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 53.9 MB/s eta 0:00:00


In [5]:
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. 임베딩 모델 로드
model = SentenceTransformer('jhgan/ko-sbert-nli')

# 2. RAG용 청크 데이터 불러오기
rag_chunks_path = "/content/drive/MyDrive/RAG/data/rag_chunks.json"
docs = []
with open(rag_chunks_path, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

In [6]:
# 3. 텍스트 추출
texts = [doc["context"] for doc in docs]

In [7]:
# 4. 임베딩 생성

# embeddings = model.encode(texts, show_progress_bar=True)
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

Batches:   0%|          | 0/251 [00:00<?, ?it/s]

In [8]:
# 5. FAISS 인덱스 생성

# 1) L2 정규화 (Cosine similarity 용)
faiss.normalize_L2(embeddings)

# 2) Inner Product 인덱스 생성 (벡터 검색 시 코사인 유사도 사용)
flat_index = faiss.IndexFlatIP(embeddings.shape[1])

# 3) IndexIDMap으로 래핑해서, 고유 ID를 붙여 추가
index = faiss.IndexIDMap(flat_index)

#    └─ docs 리스트 순서대로 0,1,2,…를 ID로 사용
ids = np.array([int(doc['id'].split('_')[-1]) for doc in docs], dtype='int64')
index.add_with_ids(embeddings, ids)

'''
이후에 이 id를 가지고 원본 문서의 id나 context에 바로 매핑할 수 있음

_, retrieved_ids = index.search(query_embedding, k)
for rid in retrieved_ids[0]:
    print(docs[rid]['id'], docs[rid]['context'])

'''

# 6. 인덱스 저장
faiss.write_index(index, "/content/drive/MyDrive/RAG/rag_faiss.index")


In [9]:
import faiss

# 인덱스 로드
index_path = "/content/drive/MyDrive/RAG/rag_faiss.index"
index = faiss.read_index(index_path)

# 벡터 개수, 차원 확인
print("벡터 개수:", index.ntotal)
print("벡터 차원:", index.d)


벡터 개수: 8026
벡터 차원: 768


In [10]:
import json

texts_path = "/content/drive/MyDrive/RAG/rag_texts.json"

# 텍스트+메타데이터 로딩
with open(texts_path, "r", encoding="utf-8") as f:
    docs = json.load(f)

print(f"문서 개수: {len(docs)}")
print("예시 문서:")
print(json.dumps(docs[0], indent=2, ensure_ascii=False))


문서 개수: 8026
예시 문서:
{
  "id": "entry_0",
  "context": "[질문] 지금 제일 가보고 싶은 회사는\n[답변] 제가 이 회사에 지원한 이유는 지금 가장 가고 싶은 회사가 이 회사이기 때문입니다. 이 회사에서 근무를 하면서 더 많은 능력을 키우고 더 많은 경험을 통해서 제가 가고 싶었던 이 회사 회사를 회사에 좀 더 기여할 수 있도록 많은 노력을 하려고 하고 있습니다.",
  "metadata": {
    "occupation": "ARD",
    "gender": "FEMALE",
    "experience": "NEW",
    "date": "20230116",
    "emotion": [
      "p-interest",
      "p-interest"
    ],
    "intent_category": [
      "etc"
    ],
    "summary": "제가 이 회사에 지원한 이유는 가장 가고 싶은 회사이기 때문입니다. 회사에서 근무하며 회사에 기여할 수 있도록 노력을 하겠습니다."
  }
}
